In [ ]:
import torch
import torch.autograd as autograd         # computation graph
from torch import Tensor                  # tensor node in the computation graph
import torch.nn as nn                     # neural networks
import torch.optim as optim               # optimizers e.g. gradient descent, ADAM, etc.

import numpy as np
import math
import time
from timeit import default_timer as timer

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.colors as cls

# Device configuration
device = torch.device('cpu')


torch.set_default_dtype(torch.float64)

class SNN(nn.Module):

    def __init__(self, layers1,layers2, device):
        super().__init__() #call __init__ from parent class
        self.layers1 = layers1
        self.layers2 = layers2
        self.device = device
        self.activation = nn.Tanh()
        self.loss_function = nn.MSELoss(reduction ='mean')
        self.linears1 = nn.ModuleList([
            nn.Linear(layers1[i], layers1[i+1]).to(dtype=torch.float64) 
            for i in range(len(layers1) - 1)
        ])
        self.linears2 = nn.ModuleList([
            nn.Linear(layers2[i], layers2[i+1]).to(dtype=torch.float64) 
            for i in range(len(layers2) - 1)
        ])
        self.M = layers1[-1]
        self.omega = torch.ones((self.M, 1), dtype=torch.float64).to(device)
        self.alpha = 1e-2
        self.a = 0
        self.b = 1
        
        self.D = 1.0         
        self.gamma = 1.0      
        self.beta1 = 1.0     
        self.beta2 = 1.0      

        

        'Xavier Normal Initialization'
        for i1 in range(len(layers1)-1):
            nn.init.xavier_normal_(self.linears1[i1].weight.data, gain=1.0)
            nn.init.zeros_(self.linears1[i1].bias.data)
            
        for i2 in range(len(layers2)-1):
            nn.init.xavier_normal_(self.linears2[i2].weight.data, gain=1.0)
            nn.init.zeros_(self.linears2[i2].bias.data)
            
    
        

    'foward pass'
    def forward_y(self,x):

        if torch.is_tensor(x) != True:
            x = torch.from_numpy(x).to(dtype=torch.float64).to(self.device)
        

        l_x = torch.prod((x - self.a) * (self.b - x), 1)[:,None]
        
        #x = torch.sin( math.pi * x)
        for i in range(len(self.layers1)-2):
            z = self.linears1[i](x)
            x = torch.sin(z)
        
        x = self.linears1[-1](x) #+ x_sum

        output = l_x *  x


        return output
    
    def forward_p(self,x):

        if torch.is_tensor(x) != True:
            x = torch.from_numpy(x).to(dtype=torch.float64).to(self.device)
        

        l_x = torch.prod((x - self.a) * (self.b - x), 1)[:,None]
        
        #x = torch.sin( math.pi * x)
        for i in range(len(self.layers2)-2):
            z = self.linears2[i](x)
            x = torch.sin(z)
        
        x = self.linears2[-1](x) #+ x_sum

        output = l_x *  x


        return output
                                
    def exact_solution(self, x):
    
        x1, x2 = x[:, 0:1], x[:, 1:2]


        y_exact = torch.sin(3 * math.pi * x1) * torch.sin(3 * math.pi* x2)
        u_exact = 2 * math.pow(math.pi,2) * y_exact
        p_exact = - self.alpha * u_exact

        grad_y_x1 = 3 * math.pi * torch.cos(3 * math.pi * x1) * torch.sin(3 * math.pi* x2)
        grad_y_x2 = 3 * math.pi * torch.cos(3 * math.pi * x2) * torch.sin(3 * math.pi* x1)

        grad_p_x1 = - self.alpha * 2 * math.pow(math.pi,2) * grad_y_x1
        grad_p_x2 = - self.alpha * 2 * math.pow(math.pi,2) * grad_y_x2

        
        laplace_y = - 18 * math.pi**2 *  y_exact
        laplace_p = - 2 * self.alpha * math.pi**2 * laplace_y

        y_d = self.D * laplace_p + self.beta1 * grad_p_x1 + self.beta2 * grad_p_x2 + y_exact - p_exact 
       
        f = - self.D * laplace_y + self.beta1 * grad_y_x1 + self.beta2 * grad_y_x2 - u_exact + y_exact
        return u_exact, y_exact, p_exact, y_d, f


    
    
    def loss(self, x):
        g = x
        g.requires_grad = True

        fi_y = self.forward_y(g)
        fi_p = self.forward_p(g)

        y = fi_y @ self.omega
        p = fi_p @ self.omega
        u = -1 / self.alpha * p

        # ∇y
        y_grad = autograd.grad(y, g, torch.ones_like(y), retain_graph=True, create_graph=True)[0]
        y_x1, y_x2 = y_grad[:, [0]], y_grad[:, [1]]

        # Δy
        y_x1x1 = autograd.grad(y_x1, g, torch.ones_like(y_x1), retain_graph=True, create_graph=True)[0][:, [0]]
        y_x2x2 = autograd.grad(y_x2, g, torch.ones_like(y_x2), retain_graph=True, create_graph=True)[0][:, [1]]
        laplace_y = y_x1x1 + y_x2x2

        # ∇p
        p_grad = autograd.grad(p, g, torch.ones_like(p), retain_graph=True, create_graph=True)[0]
        p_x1, p_x2 = p_grad[:, [0]], p_grad[:, [1]]

        # Δp
        p_x1x1 = autograd.grad(p_x1, g, torch.ones_like(p_x1), retain_graph=True, create_graph=True)[0][:, [0]]
        p_x2x2 = autograd.grad(p_x2, g, torch.ones_like(p_x2), retain_graph=True, create_graph=True)[0][:, [1]]
        laplace_p = p_x1x1 + p_x2x2
    
        
        u_exact, y_exact, p_exact, y_d, f = self.exact_solution(x)

       
        convection_y = self.beta1 * y_x1 + self.beta2 * y_x2
        PDE_residual = -self.D * laplace_y + convection_y + self.gamma * y - u - f
        #PDE_residual = -self.D * laplace_y  + y_x1 + self.gamma * y - u - f

        convection_p = self.beta1 * p_x1 + self.beta2 * p_x2
        adjoint_residual = -self.D * laplace_p - convection_p + self.gamma * p - (y - y_d)
       

       
        loss_p = self.loss_function(adjoint_residual, torch.zeros_like(adjoint_residual))
        loss_PDE = self.loss_function(PDE_residual, torch.zeros_like(PDE_residual))

        return loss_p,loss_PDE

                                
                                
                                
    
                                
                                
    def train_model_adam(self, optimizer, x, n_epoch):
        if not torch.is_tensor(x):
            x = torch.from_numpy(x).to(dtype=torch.float64).to(self.device)

        iter_s = 0
        epsilon = 1e-3  
        gamma = 0.9
        eps = 1e-8
        w_min = 1
        w_max = 1.1
        beta = 0.5

        ema_p, ema_PDE = 1.0, 1.0

        while iter_s < n_epoch:
            optimizer.zero_grad()
            loss_p, loss_PDE = self.loss(x)

            
            val_p = float(loss_p.detach())
            val_PDE = float(loss_PDE.detach())

            
            new_ema_p = gamma * ema_p + (1 - gamma) * val_p
            new_ema_PDE = gamma * ema_PDE + (1 - gamma) * val_PDE

         
            r1 = max((ema_p - new_ema_p) / (ema_p + eps), 0.0)
            r2 = max((ema_PDE - new_ema_PDE) / (ema_PDE + eps), 0.0)

           
            raw_w1 = 1.0 / ((r1 + eps) ** beta)
            raw_w2 = 1.0 / ((r2 + eps) ** beta)

          
            w1 = raw_w1 / (raw_w1 + raw_w2)

           
            w1 = w_min + (w_max - w_min) * w1
            w2 = 2.1 - w1

     
            loss = w1*loss_p + w2*loss_PDE
            loss1 = loss_p + loss_PDE

         
            if iter_s == 0:
                initial_loss = float(loss1.detach())
                if initial_loss < 1e-13:   # 太小直接停止
                    print("Stop")
                    break

            
            if float(loss1.detach()) / initial_loss < epsilon:
                print("Stop:", iter_s,
                      float(loss_p.detach()), float(loss_PDE.detach()),
                      float(loss1.detach()),w1,w2)
                break

           
            loss.backward()
            optimizer.step()

         
            ema_p, ema_PDE = new_ema_p, new_ema_PDE

       
            if iter_s == 0 or iter_s % 100 == 0:
                print(iter_s,
                      float(loss_p.detach()), float(loss_PDE.detach()),
                      float(loss1.detach()),w1,w2)

            iter_s += 1

In [ ]:
# === Sampling ===
def sample_domain(n=32):
    x = np.linspace(0, 1, n)
    y = np.linspace(0, 1, n)
    xx, yy = np.meshgrid(x, y)
    xy = np.stack([xx.ravel(), yy.ravel()], axis=-1)
    x_train = torch.tensor(xy, dtype=torch.float64)
    return x_train
def sample_boundary(n=32):
    x = np.linspace(0, 1, n)
    y = np.linspace(0, 1, n)
    points = []
    for xi in x:
        points.append([xi, 0])
        points.append([xi, 1])
    for yi in y:
        points.append([0, yi])
        points.append([1, yi])
    x_b_train = torch.tensor(points, dtype=torch.float64)
    return x_b_train

M = 200
layers1 = np.array([2, 100, 100, 100, 100,  M]) 
layers2 = np.array([2, 100, 100, 100, 100,  M])

SNN = SNN(layers1, layers2, device)

SNN.to(device)
#print(PINN)
optimizer = torch.optim.Adam(SNN.parameters(), lr=0.001)
x_train = sample_domain(n=32)
x_b_train = sample_boundary(n=32)
start = timer()
SNN.train_model_adam(optimizer, x_train, 3e4)

In [ ]:
from scipy.optimize import least_squares


D = SNN.D
beta1, beta2 = SNN.beta1, SNN.beta2
gamma = SNN.gamma
alpha = SNN.alpha
epoch = 20



# === 1 ===
phi_y = SNN.forward_y(x_train).detach().cpu().numpy()
phi_y_bc = SNN.forward_y(x_b_train).detach().cpu().numpy()

phi_p = SNN.forward_p(x_train).detach().cpu().numpy()
phi_p_bc = SNN.forward_p(x_b_train).detach().cpu().numpy()

# === 2 ===
def compute_phi_derivatives(forward_fn, x, M):
    Lap, Grad_x, Grad_y = [], [], []
    for j in range(M):
        phi_j = forward_fn(x)[:, j:j+1]
        grads = torch.autograd.grad(phi_j, x, torch.ones_like(phi_j), create_graph=True)[0]
        dx, dy = grads[:, [0]], grads[:, [1]]

        d2x = torch.autograd.grad(dx, x, torch.ones_like(dx), create_graph=True)[0][:, [0]]
        d2y = torch.autograd.grad(dy, x, torch.ones_like(dy), create_graph=True)[0][:, [1]]
        lap = (d2x + d2y).detach().cpu().numpy()
        grad_x = dx.detach().cpu().numpy()
        grad_y = dy.detach().cpu().numpy()

        Lap.append(lap)
        Grad_x.append(grad_x)
        Grad_y.append(grad_y)

    return (
        np.concatenate(Lap, axis=1),
        np.concatenate(Grad_x, axis=1),
        np.concatenate(Grad_y, axis=1)
    )

Lap_y, Grad_phi_y_x, Grad_phi_y_y = compute_phi_derivatives(SNN.forward_y, x_train, M)
Lap_p, Grad_phi_p_x, Grad_phi_p_y = compute_phi_derivatives(SNN.forward_p, x_train, M)

# === 3 ===
_, y_exact, p_exact, y_d, f = SNN.exact_solution(x_train)
_, yb_exact, pb_exact, _, _ = SNN.exact_solution(x_b_train)

f_np = f.detach().cpu().numpy()
y_d_np = y_d.detach().cpu().numpy()
yb = yb_exact.detach().cpu().numpy()
pb = pb_exact.detach().cpu().numpy()

# === 4 ===
A_y = phi_y
A_y_bc = phi_y_bc
A_p = phi_p
A_p_bc = phi_p_bc

# === 5 ===
omega_y = np.ones((M, 1))
omega_p = np.ones((M, 1))

# === 6 ===
for i in range(epoch):

    # === Step 1 ===
    def residual_y(omega_y_np):
        omega_y = omega_y_np[:, None]
        y = A_y @ omega_y
        p = A_p @ omega_p
        lap_y = Lap_y @ omega_y
        grad_x = Grad_phi_y_x @ omega_y
        grad_y = Grad_phi_y_y @ omega_y
        convection = beta1 * grad_x + beta2 * grad_y
        r_pde = -D * lap_y + convection + gamma * y + (1 / alpha) * p - f_np
        #r_pde = -D * lap_y + grad_x + gamma * y + (1 / alpha) * p - f_np
        r_bc = A_y_bc @ omega_y - yb
        return np.vstack([r_pde, r_bc]).ravel()

    res_y = least_squares(residual_y, omega_y.ravel(), method='trf')
    omega_y = res_y.x[:, None]

    # === Step 2 ===
    def residual_p(omega_p_np):
        omega_p = omega_p_np[:, None]
        p = A_p @ omega_p
        y = A_y @ omega_y
        lap_p = Lap_p @ omega_p
        grad_x = Grad_phi_p_x @ omega_p
        grad_y = Grad_phi_p_y @ omega_p
        convection = beta1 * grad_x + beta2 * grad_y
        r_pde = -D * lap_p - convection + gamma * p - (y - y_d_np)
        #r_pde = -D * lap_p - grad_x + gamma * p - (y - y_d_np)
        r_bc = A_p_bc @ omega_p - pb
        return np.vstack([r_pde, r_bc]).ravel()

    res_p = least_squares(residual_p, omega_p.ravel(), method='trf')
    omega_p = res_p.x[:, None]

    print(f"🔁 Iter {i+1}/{epoch}, state: {res_y.cost:.2e}, adjoint {res_p.cost:.2e}")

end = timer()
print("✅ Total time: {:.2f}s".format(end - start))


In [ ]:
omega_p = torch.tensor(omega_p, dtype=torch.float64).to(device)
omega_y = torch.tensor(omega_y, dtype=torch.float64).to(device)
def test(x_test):

        if torch.is_tensor(x_test) != True:
            x_test = torch.from_numpy(x_test).to(dtype=torch.float64).to(self.device)
            
        u, y, p, y_d,_ = SNN.exact_solution(x_test)

        phi_pred_y = SNN.forward_y(x_test)
        phi_pred_p = SNN.forward_p(x_test)
        y_pred = phi_pred_y @ omega_y
        p_pred = phi_pred_p @ omega_p
        u_pred = -1/SNN.alpha * p_pred
    
        error_y = torch.linalg.norm((y - y_pred),2)/torch.linalg.norm(y,2)
        y_pred = y_pred.cpu().detach().numpy()
        
        error_u = torch.linalg.norm((u - u_pred),2)/torch.linalg.norm(u,2)
        u_pred = u_pred.cpu().detach().numpy()
        
    
        return error_y, y_pred, y,error_u, u_pred, u

x_test = sample_domain(n=100).to(device)
error_y, y_pred, y ,error_u, u_pred, u= test(x_test)
print('Test Error y: {:.2e}'.format(error_y))
print('Test Error u: {:.2e}'.format(error_u))